In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from functools import reduce

In [2]:
train = pd.read_csv('../../data/metadata/train_metadata.csv')
test  = pd.read_csv('../../data/metadata/test_metadata.csv')
folds = pd.read_csv('../../data/combined/data.csv')[['image_id','fold']]
folds.columns = ['image_name','fold']
train = train.merge(folds, on='image_name')
train['fold'] = train['fold'].map(lambda x : np.random.randint(0,100))
train.shape, test.shape

((32691, 11), (10982, 7))

In [3]:
train['num'] = train.groupby(['patient_id'])['image_name'].transform('count')
train['sex'] = train['sex'].fillna('na')
train['age_approx'] = train['age_approx'].fillna(0)
train['anatom_site_general_challenge'] = train['anatom_site_general_challenge'].fillna('na')

In [4]:
test['num'] = test.groupby(['patient_id'])['image_name'].transform('count')
test['sex'] = test['sex'].fillna('na')
test['age_approx'] = test['age_approx'].fillna(0)
test['anatom_site_general_challenge'] = test['anatom_site_general_challenge'].fillna('na')

In [5]:
train.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target,w,h,fold,num
0,ISIC_2637011,IP_7279968,male,45.0,head/neck,unknown,benign,0,6000.0,4000.0,65,115
1,ISIC_0015719,IP_3075186,female,45.0,upper extremity,unknown,benign,0,6000.0,4000.0,14,24
2,ISIC_0052212,IP_2842074,female,50.0,lower extremity,nevus,benign,0,1872.0,1053.0,54,5
3,ISIC_0068279,IP_6890425,female,45.0,head/neck,unknown,benign,0,1872.0,1053.0,63,22
4,ISIC_0074268,IP_8723313,female,55.0,upper extremity,unknown,benign,0,6000.0,4000.0,65,20


In [6]:
test.head()

,image_name,patient_id,sex,age_approx,anatom_site_general_challenge,w,h,num
0,ISIC_0052060,IP_3579794,male,70.0,na,6000.0,4000.0,240
1,ISIC_0052349,IP_7782715,male,40.0,lower extremity,6000.0,4000.0,46
2,ISIC_0058510,IP_7960270,female,55.0,torso,6000.0,4000.0,28
3,ISIC_0073313,IP_6375035,female,50.0,torso,6000.0,4000.0,38
4,ISIC_0073502,IP_0589375,female,45.0,lower extremity,1920.0,1080.0,29


In [7]:
FEATURES = ['sex','age_approx','anatom_site_general_challenge','num','w','h']
L = 50.
M = train.target.mean()
L,M

(50.0, 0.017772475604906548)

In [8]:
def smoothEncode(fold):
    train_subset = train[train['fold'] != fold].copy()
    valid_subset = train[train['fold'] == fold].copy()
    test_subset = test.copy()
    encode = train_subset.groupby(FEATURES)['target'].agg(['mean','count']).reset_index()
    encode['score'] = ((encode['mean']*encode['count']) + (M*L))/(encode['count'] + L)
    valid_subset = valid_subset.merge(encode, on=FEATURES, how='left')
    test_subset = test_subset.merge(encode, on=FEATURES, how='left')
    valid_subset['score'] = valid_subset['score'].fillna(M)
    test_subset['score'] = test_subset['score'].fillna(M)
    valid_subset = valid_subset[['image_name','target','score']]
    test_subset = test_subset[['image_name','score']]
    return valid_subset, test_subset

In [9]:
valid_dataset, test_dataset = [], []

In [10]:
for idx in range(100):
    val_tmp, test_tmp = smoothEncode(idx)
    valid_dataset.append(val_tmp)
    test_dataset.append(test_tmp)

In [11]:
train = reduce(lambda x, y : x.append(y), valid_dataset)
train = train.groupby(['image_name','target']).mean().reset_index()

In [12]:
roc_auc_score(train.target, train.score)

0.7892955637114458

In [13]:
test = reduce(lambda x, y : x.append(y), test_dataset)
test = test.groupby(['image_name']).mean().reset_index()

In [14]:
train = train[['image_name','score']]
test = test[['image_name','score']]

In [15]:
train.columns = ['image_name','score']
test.columns = ['image_name','target']

In [16]:
train.to_csv('../../score/metadata_train_score.csv', index=False)
test.to_csv('../../score/metadata_test_score.csv', index=False)

In [17]:
train.shape, test.shape

((32691, 2), (10982, 2))